# Models with other features

In [1]:
# ============================================
# 0. Imports & basic config
# ============================================
import pandas as pd
import numpy as np

# Models
import xgboost as xgb
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [2]:
# ============================================
# 1. Load data
# ============================================
train = pd.read_parquet('../data/model/final_train.parquet')
val = pd.read_parquet('../data/model/final_val.parquet')
test = pd.read_parquet('../data/model/final_test.parquet')

print("Train Date range:", train["Date"].min(), "to", train["Date"].max())
print("Val Date range:  ", val["Date"].min(),   "to", val["Date"].max())
print("Test Date range: ", test["Date"].min(),  "to", test["Date"].max())

Train Date range: 2016-01-04 00:00:00 to 2021-12-30 00:00:00
Val Date range:   2022-01-03 00:00:00 to 2022-12-30 00:00:00
Test Date range:  2023-01-03 00:00:00 to 2023-12-29 00:00:00


In [3]:
X_train = train.loc[:, train.columns != "return_next_day"]
X_val = val.loc[:, val.columns != "return_next_day"]
X_test = test.loc[:, test.columns != "return_next_day"]
X_full = pd.concat([X_train, X_val, X_test], axis=0).reset_index(drop=True)

y_train = train[['Date', 'tic', 'return_next_day']]
y_val = val[['Date', 'tic', 'return_next_day']]
y_test = test[['Date', 'tic', 'return_next_day']]
y_full = pd.concat([y_train, y_val, y_test], axis=0).reset_index(drop=True)

In [4]:
# ============================================
# 2. Directional accuracy helper
# ============================================
def directional_accuracy(y_true, y_pred):
    """
    y_true, y_pred: 1D numpy arrays of returns
    Returns fraction of times sign(pred) == sign(true).
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return (np.sign(y_true) == np.sign(y_pred)).mean()

In [5]:
# ============================================
# 3. Targets (1D vectors)
# ============================================
y_train_vec = train["return_next_day"].values.astype("float32")
y_val_vec   = val["return_next_day"].values.astype("float32")
y_test_vec  = test["return_next_day"].values.astype("float32")

In [6]:
sp500_info = pd.read_csv('../data/sp500_companies.csv')
sp500_info.head()

,ticker,company_name,sector,subsector,cik
0,MMM,3M,Industrials,Industrial Conglomerates,66740
1,AOS,A. O. Smith,Industrials,Building Products,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1800
3,ABBV,AbbVie,Health Care,Biotechnology,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,1467373


In [7]:
sector_info = (
    sp500_info[["ticker", "sector", "subsector"]]
    .rename(columns={"ticker": "tic"})
)

In [8]:
# ============================================
# 4. Build DOW / MONTH / SECTOR one-hot features
#    (DMS = Day-of-week + Month + Sector)
# ============================================

# Ensure Date is datetime
for df in [train, val, test]:
    df["Date"] = pd.to_datetime(df["Date"])

# We will create one-hot columns on the concatenated frame
train["split"] = "train"
val["split"]   = "val"
test["split"]  = "test"

full = pd.concat([train, val, test], axis=0, ignore_index=True)
full = full.merge(sector_info, on="tic", how="left")

# --- Raw categorical keys for DMS ---
full["day_of_week"] = full["Date"].dt.dayofweek   # 0=Mon,...,4=Fri (likely only 0-4)
full["month"]       = full["Date"].dt.month       # 1..12
# Assume 'sector' already exists in your data
# If not, you’d merge it in from a sector mapping before this step.

# One-hot encode these three
dms_cats = ["day_of_week", "month", "sector"]
full_dms = pd.get_dummies(
    full[dms_cats],
    columns=dms_cats,
    prefix=dms_cats,
    drop_first=True,      # gives 4 DOW, 11 MONTH, 10 SECTOR (total 25)
)

print("Full DMS feature matrix shape:", full_dms.shape)
print("Example DMS columns:", full_dms.columns[:10].tolist())

Full DMS feature matrix shape: (979133, 25)
Example DMS columns: ['day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7']


In [9]:
# ============================================
# 5. Split DMS features back into train/val/test
# ============================================

# Align index with `full`
full_dms.index = full.index

mask_train = full["split"] == "train"
mask_val   = full["split"] == "val"
mask_test  = full["split"] == "test"

X_train_dms = full_dms[mask_train].reset_index(drop=True)
X_val_dms   = full_dms[mask_val].reset_index(drop=True)
X_test_dms  = full_dms[mask_test].reset_index(drop=True)

print("Num DOW cols:   ", sum(c.startswith("day_of_week_") for c in X_train_dms.columns))
print("Num MONTH cols: ", sum(c.startswith("month_") for c in X_train_dms.columns))
print("Num SECTOR cols:", sum(c.startswith("sector_") for c in X_train_dms.columns))
print("Total DMS cols: ", X_train_dms.shape[1])

print("X_train_dms shape:", X_train_dms.shape)
print("X_val_dms shape:  ", X_val_dms.shape)
print("X_test_dms shape: ", X_test_dms.shape)

Num DOW cols:    4
Num MONTH cols:  11
Num SECTOR cols: 10
Total DMS cols:  25
X_train_dms shape: (729659, 25)
X_val_dms shape:   (124747, 25)
X_test_dms shape:  (124727, 25)


In [10]:
# For tree models we can use float32
X_train_dms_32 = X_train_dms.astype("float32")
X_val_dms_32   = X_val_dms.astype("float32")
X_test_dms_32  = X_test_dms.astype("float32")

In [11]:
# ============================================
# 6. LightGBM-safe column names (for DMS)
#    (spaces, slashes, etc. -> '_')
# ============================================
import re

def make_lgbm_safe_colnames(cols):
    """
    Replace any non [0-9A-Za-z_] characters with '_'
    to keep LightGBM happy.
    """
    new_cols = []
    for c in cols:
        safe = re.sub(r"[^0-9A-Za-z_]+", "_", c)
        new_cols.append(safe)
    return new_cols

orig_dms_cols = list(X_train_dms.columns)
safe_dms_cols = make_lgbm_safe_colnames(orig_dms_cols)
dms_col_map   = dict(zip(orig_dms_cols, safe_dms_cols))

X_train_dms_lgb = X_train_dms.rename(columns=dms_col_map)
X_val_dms_lgb   = X_val_dms.rename(columns=dms_col_map)
X_test_dms_lgb  = X_test_dms.rename(columns=dms_col_map)

print("Example LGBM-safe cols:", list(X_train_dms_lgb.columns[:10]))

Example LGBM-safe cols: ['day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7']


In [12]:
# 7.1 Best XGBoost params on structure-like features
best_xgb_dms_params = {
    "max_depth":        4,
    "learning_rate":    0.03,   # <-- UPDATED
    "n_estimators":     800,    # <-- UPDATED
    "subsample":        0.70,
    "colsample_bytree": 0.70,   # <-- UPDATED
    "tree_method":      "hist",
    "n_jobs":           -1,
    "random_state":     RANDOM_SEED,
}

xgb_dms_model = xgb.XGBRegressor(**best_xgb_dms_params)

### Phase 1 results

In [13]:
# %%
# 4. Best XGBoost params on DOW+MONTH+SECTOR (no tic)

best_xgb_dms_params = {
    "max_depth":        4,
    "learning_rate":    0.03,
    "n_estimators":     800,
    "subsample":        0.70,
    "colsample_bytree": 0.70,
    "tree_method":      "hist",
    "n_jobs":           -1,
    "random_state":     RANDOM_SEED,
}

xgb_dms_model = xgb.XGBRegressor(**best_xgb_dms_params)

# (Optional: quick sanity fit/eval; or leave for later experiments)
xgb_dms_model.fit(X_train_dms_32, y_train_vec, eval_set=[(X_val_dms_32, y_val_vec)], verbose=False)

val_pred_xgb  = xgb_dms_model.predict(X_val_dms_32)
test_pred_xgb = xgb_dms_model.predict(X_test_dms_32)

print("XGBoost (DMS only):")
print("  Val DA: ", directional_accuracy(y_val_vec,  val_pred_xgb))
print("  Test DA:", directional_accuracy(y_test_vec, test_pred_xgb))

XGBoost (DMS only):
  Val DA:  0.5169182425228663
  Test DA: 0.5280011545214749


In [14]:
# %%
# 6. Build DMS + tic matrices for CatBoost

# Numeric DMS features (same as before)
X_train_dms_cb = X_train_dms_32.copy()
X_val_dms_cb   = X_val_dms_32.copy()
X_test_dms_cb  = X_test_dms_32.copy()

# Add tic as a separate column
X_train_dms_tic = pd.concat([train[["tic"]].reset_index(drop=True), X_train_dms_cb.reset_index(drop=True)], axis=1)
X_val_dms_tic   = pd.concat([val[["tic"]].reset_index(drop=True),   X_val_dms_cb.reset_index(drop=True)],   axis=1)
X_test_dms_tic  = pd.concat([test[["tic"]].reset_index(drop=True),  X_test_dms_cb.reset_index(drop=True)],  axis=1)

print("DMS+tic train shape:", X_train_dms_tic.shape)

# index of 'tic' for CatBoost categorical_features
cat_feature_idx = [X_train_dms_tic.columns.get_loc("tic")]
print("Categorical feature indices:", cat_feature_idx)

DMS+tic train shape: (729659, 26)
Categorical feature indices: [0]


In [15]:
# %%
# 7. Best CatBoost params for DMS + tic (from tuning)
# depth=4, learning_rate=0.01, l2_leaf_reg=5, bagging_temperature=1, border_count=128, best_iter ~ 597

best_cb_dms_tic_params = {
    "depth":              4,
    "learning_rate":      0.01,
    "l2_leaf_reg":        5,
    "bagging_temperature": 1,
    "border_count":       128,
    "iterations":         597,   # a bit above best_iter, still with overfitting control
    "loss_function":      "RMSE",
    "random_seed":        RANDOM_SEED,
    "verbose":            False,
}

cat_dms_tic_model = CatBoostRegressor(**best_cb_dms_tic_params)

cat_dms_tic_model.fit(
    X_train_dms_tic,
    y_train_vec,
    eval_set=(X_val_dms_tic, y_val_vec),
    cat_features=cat_feature_idx,
    verbose=False,
)

pred_val_cb  = cat_dms_tic_model.predict(X_val_dms_tic)
pred_test_cb = cat_dms_tic_model.predict(X_test_dms_tic)

val_DA_cb  = directional_accuracy(y_val_vec,  pred_val_cb)
test_DA_cb = directional_accuracy(y_test_vec, pred_test_cb)

print("CatBoost (DMS + tic):")
print("  Val DA: ", val_DA_cb)
print("  Test DA:", test_DA_cb)

CatBoost (DMS + tic):
  Val DA:  0.5178962219532334
  Test DA: 0.5354093339854241


In [16]:
# %%
# 5. Best LightGBM params on DOW+MONTH+SECTOR (no tic)

best_lgb_dms_params = {
    "num_leaves":       63,
    "max_depth":        -1,
    "learning_rate":    0.03,
    "n_estimators":     500,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq":     1,
}

lgb_dms_model = LGBMRegressor(
    objective="regression",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    **best_lgb_dms_params,
)

# Early stopping on validation RMSE; LightGBM uses its own best_iteration_
callbacks = [early_stopping(stopping_rounds=50, verbose=False)]

lgb_dms_model.fit(
    X_train_dms_lgb,
    y_train_vec,
    eval_set=[(X_val_dms_lgb, y_val_vec)],
    eval_metric="rmse",
    callbacks=callbacks,
)

best_iter_lgb = lgb_dms_model.best_iteration_ or best_lgb_dms_params["n_estimators"]

val_pred_lgb  = lgb_dms_model.predict(X_val_dms_lgb,  num_iteration=best_iter_lgb)
test_pred_lgb = lgb_dms_model.predict(X_test_dms_lgb, num_iteration=best_iter_lgb)

print("LightGBM (DMS only):")
print("  Val DA: ", directional_accuracy(y_val_vec,  val_pred_lgb))
print("  Test DA:", directional_accuracy(y_test_vec, test_pred_lgb))

[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing row-wise multi-threadi

In [17]:
# %%
results_phase1 = pd.DataFrame([
    {
        "model": "XGBoost (DMS)",
        "val_DA": directional_accuracy(y_val_vec,  val_pred_xgb),
        "test_DA": directional_accuracy(y_test_vec, test_pred_xgb),
    },
    {
        "model": "LightGBM (DMS)",
        "val_DA": directional_accuracy(y_val_vec,  val_pred_lgb),
        "test_DA": directional_accuracy(y_test_vec, test_pred_lgb),
    },
    {
        "model": "CatBoost (DMS + tic)",
        "val_DA": val_DA_cb,
        "test_DA": test_DA_cb,
    },
])

results_phase1 = results_phase1.sort_values('val_DA', ascending=False)
results_phase1

,model,val_DA,test_DA
1,LightGBM (DMS),0.519908,0.533870
2,CatBoost (DMS + tic),0.517896,0.535409
0,XGBoost (DMS),0.516918,0.528001


### Phase 2 - adding other features

In [18]:
import pandas as pd

# --- Make sure sector is available on X_* ---
# sector_info: columns ["tic", "sector", "subsector"]
X_train = X_train.merge(sector_info, on="tic", how="left")
X_val   = X_val.merge(sector_info, on="tic", how="left")
X_test  = X_test.merge(sector_info, on="tic", how="left")

# --- Create raw DOW and MONTH columns ---
for df in (X_train, X_val, X_test):
    df["Date"] = pd.to_datetime(df["Date"])
    df["day_of_week"] = df["Date"].dt.dayofweek  # 0–6 (you’ll only have 0–4)
    df["month"]       = df["Date"].dt.month      # 1–12

dms_cats = ["day_of_week", "month", "sector"]

# One-hot on TRAIN: defines canonical DMS columns
X_train_dms = pd.get_dummies(
    X_train[dms_cats],
    columns=dms_cats,
    prefix=dms_cats,
    drop_first=True,   # 4 DOW, 11 MONTH, ~10 SECTOR => ~25 cols
)
dms_cols = X_train_dms.columns.tolist()

# One-hot on VAL/TEST, align to TRAIN's DMS columns
X_val_dms = pd.get_dummies(
    X_val[dms_cats],
    columns=dms_cats,
    prefix=dms_cats,
    drop_first=True,
).reindex(columns=dms_cols, fill_value=0)

X_test_dms = pd.get_dummies(
    X_test[dms_cats],
    columns=dms_cats,
    prefix=dms_cats,
    drop_first=True,
).reindex(columns=dms_cols, fill_value=0)

print("Num DOW cols:   ", sum(c.startswith("day_of_week_") for c in dms_cols))
print("Num MONTH cols: ", sum(c.startswith("month_") for c in dms_cols))
print("Num SECTOR cols:", sum(c.startswith("sector_") for c in dms_cols))
print("Total DMS cols: ", len(dms_cols))

print("X_train_dms shape:", X_train_dms.shape)
print("X_val_dms shape:  ", X_val_dms.shape)
print("X_test_dms shape: ", X_test_dms.shape)

Num DOW cols:    4
Num MONTH cols:  11
Num SECTOR cols: 10
Total DMS cols:  25
X_train_dms shape: (729659, 25)
X_val_dms shape:   (124747, 25)
X_test_dms shape:  (124727, 25)


In [19]:
# Attach DMS columns back to X_train / X_val / X_test
X_train = pd.concat([X_train.reset_index(drop=True), X_train_dms.reset_index(drop=True)], axis=1)
X_val   = pd.concat([X_val.reset_index(drop=True),   X_val_dms.reset_index(drop=True)],   axis=1)
X_test  = pd.concat([X_test.reset_index(drop=True),  X_test_dms.reset_index(drop=True)],  axis=1)

In [20]:
# Recompute dms_cols from the new X_train (just to be safe)
dow_cols    = [c for c in X_train.columns if c.startswith("day_of_week_")]
month_cols  = [c for c in X_train.columns if c.startswith("month_")]
sector_cols = [c for c in X_train.columns if c.startswith("sector_")]

dms_cols = dow_cols + month_cols + sector_cols

# ---- Fundamental features ----
fund_cols = [
    "sales_growth_qoq", "sales_growth_ttm",
    "asset_growth", "equity_growth",
    "roa_ttm", "roe_ttm",
    "gross_margin_ttm", "oper_margin_ttm", "net_margin_ttm",
    "log_mktcap", "bm", "earnings_yield", "cf_yield",
    "sales_yield", "div_yield",
    "leverage", "current_ratio", "cash_assets", "accruals_ta",
]

# ---- Macro features ----
macro_cols = [
    "cpi", "fedfunds", "industrial_production", "gdp",
    "retail_sales", "unemployment",
    "t10y", "t2y", "t3m", "aaa_yield",
    "vix", "sp500", "yield_spread_10y_2y",
]

# ---- Sentiment features ----
sent_cols = [
    "mean_sentiment", "max_sentiment",
    "min_sentiment", "sum_sentiment", "news_count",
]

# ---- PCA text embeddings ----
pca_cols = [c for c in X_train.columns if c.startswith("pca_emb_")]

feature_groups = {
    "DMS_only":              dms_cols,
    "DMS_plus_fundamentals": dms_cols + fund_cols,
    "DMS_plus_macro":        dms_cols + macro_cols,
    "DMS_plus_sentiment":    dms_cols + sent_cols,
    "DMS_plus_PCA":          dms_cols + pca_cols,
    "DMS_plus_all_struct":   dms_cols + fund_cols + macro_cols + sent_cols,
}

XGBoost

In [21]:
# ============================================
# 1. XGBoost helper (uses ORIGINAL column names)
# ============================================
import xgboost as xgb

# Best DMS params from your tuning:
# Val DA ≈ 0.5169, Test DA ≈ 0.5280
best_xgb_dms_params = {
    "max_depth":        4,
    "learning_rate":    0.03,   # <--- updated
    "n_estimators":     800,    # <--- updated
    "subsample":        0.70,
    "colsample_bytree": 0.70,
    "tree_method":      "hist",
    "n_jobs":           -1,
    "random_state":     42,
}

def run_xgb_on_cols(col_list, label):
    """
    Fit XGBoost with best_xgb_dms_params on the given subset of columns
    (using X_train / X_val / X_test) and return (val_DA, test_DA).
    """
    Xtr = X_train[col_list]
    Xva = X_val[col_list]
    Xte = X_test[col_list]

    model = xgb.XGBRegressor(**best_xgb_dms_params)

    model.fit(
        Xtr,
        y_train_vec,
        eval_set=[(Xva, y_val_vec)],
        verbose=False,
    )

    val_pred  = model.predict(Xva)
    test_pred = model.predict(Xte)

    val_da  = directional_accuracy(y_val_vec,  val_pred)
    test_da = directional_accuracy(y_test_vec, test_pred)

    print(f"[XGB] {label}: Val DA={val_da:.4f}, Test DA={test_da:.4f}")
    return val_da, test_da

LightGBM

In [25]:
# ============================================
# 2. LightGBM helper (uses SAFE col names)
# ============================================
from lightgbm import LGBMRegressor, early_stopping

best_lgb_dms_params = {
    "num_leaves":       63,
    "max_depth":        -1,
    "learning_rate":    0.03,
    "n_estimators":     500,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq":     1,
}

def run_lgb_on_cols(col_list, label):
    """
    Fit LightGBM on a given subset of columns in X_train / X_val / X_test
    using best_lgb_dms_params. Returns (val_DA, test_DA).
    """
    Xtr = X_train[col_list]
    Xva = X_val[col_list]
    Xte = X_test[col_list]

    model = LGBMRegressor(
        objective="regression",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        **best_lgb_dms_params,
    )

    callbacks = [early_stopping(stopping_rounds=50, verbose=False)]

    model.fit(
        Xtr,
        y_train_vec,
        eval_set=[(Xva, y_val_vec)],
        eval_metric="rmse",
        callbacks=callbacks,
    )

    best_iter = model.best_iteration_ or best_lgb_dms_params["n_estimators"]

    val_pred  = model.predict(Xva,  num_iteration=best_iter)
    test_pred = model.predict(Xte, num_iteration=best_iter)

    val_da  = directional_accuracy(y_val_vec,  val_pred)
    test_da = directional_accuracy(y_test_vec, test_pred)

    print(f"[LGBM] {label}: Val DA={val_da:.4f}, Test DA={test_da:.4f}")
    return val_da, test_da

CatBoost

In [30]:
# ============================================
# 3. CatBoost helper (DMS + optional extra groups, ALWAYS with tic)
# ============================================
from catboost import CatBoostRegressor

best_cb_params = {
    "depth":               4,
    "learning_rate":       0.01,
    "l2_leaf_reg":         5,
    "bagging_temperature": 1,
    "border_count":        128,
    "iterations":         597,
    "loss_function":       "RMSE",
    "random_seed":         42,
    "verbose":             False,
}

def run_cat_on_cols(col_list, label):
    """
    Fit CatBoost on the given subset of columns + 'tic' as a categorical feature.
    Uses best_cb_params from DMS+tic tuning.
    """
    cols_with_tic = col_list + ["tic"]

    Xtr = X_train[cols_with_tic].copy()
    Xva = X_val[cols_with_tic].copy()
    Xte = X_test[cols_with_tic].copy()

    cat_features = [cols_with_tic.index("tic")]  # index of the 'tic' column

    model = CatBoostRegressor(**best_cb_params)
    model.fit(
        Xtr,
        y_train_vec,
        eval_set=(Xva, y_val_vec),
        cat_features=cat_features,
        verbose=False,
    )

    val_pred  = model.predict(Xva)
    test_pred = model.predict(Xte)

    val_da  = directional_accuracy(y_val_vec,  val_pred)
    test_da = directional_accuracy(y_test_vec, test_pred)

    print(f"[CatBoost+ticker] {label}: Val DA={val_da:.4f}, Test DA={test_da:.4f}")
    return val_da, test_da

Run phase 2 experiment

In [31]:
results_phase2 = []

for name, cols in feature_groups.items():
    print(f"\n=== Feature group: {name} ===")

    # XGBoost (no tic)
    val_xgb, test_xgb = run_xgb_on_cols(cols, name)
    results_phase2.append({
        "model": "XGB",
        "group": name,
        "val_DA": val_xgb,
        "test_DA": test_xgb,
        "n_features": len(cols),
    })

    # LightGBM (no tic)
    val_lgb, test_lgb = run_lgb_on_cols(cols, name)
    results_phase2.append({
        "model": "LGBM",
        "group": name,
        "val_DA": val_lgb,
        "test_DA": test_lgb,
        "n_features": len(cols),
    })

    # CatBoost (with ticker as categorical)
    val_cb, test_cb = run_cat_on_cols(cols, name)
    results_phase2.append({
        "model": "CatBoost+ticker",
        "group": name,
        "val_DA": val_cb,
        "test_DA": test_cb,
        "n_features": len(cols) + 1,  # +1 for tic
    })

results_phase2_df = (
    pd.DataFrame(results_phase2)
      .sort_values(["val_DA", "test_DA"], ascending=[False, False])
)

results_phase2_df


=== Feature group: DMS_only ===
[XGB] DMS_only: Val DA=0.5169, Test DA=0.5280
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning

,model,group,val_DA,test_DA,n_features
1,LGBM,DMS_only,0.519908,0.533870,25
2,CatBoost+ticker,DMS_only,0.517896,0.535409,26
0,XGB,DMS_only,0.516918,0.528001,25
10,LGBM,DMS_plus_sentiment,0.515908,0.528129,30
3,XGB,DMS_plus_fundamentals,0.515772,0.518516,44
5,CatBoost+ticker,DMS_plus_fundamentals,0.512606,0.526101,45
11,CatBoost+ticker,DMS_plus_sentiment,0.512589,0.533261,31
9,XGB,DMS_plus_sentiment,0.512173,0.520866,30
12,XGB,DMS_plus_PCA,0.509215,0.524778,89
14,CatBoost+ticker,DMS_plus_PCA,0.507162,0.528290,90


In [34]:
results_phase2_df.sort_values('test_DA', ascending=False)

,model,group,val_DA,test_DA,n_features
2,CatBoost+ticker,DMS_only,0.517896,0.535409,26
1,LGBM,DMS_only,0.519908,0.533870,25
11,CatBoost+ticker,DMS_plus_sentiment,0.512589,0.533261,31
14,CatBoost+ticker,DMS_plus_PCA,0.507162,0.528290,90
10,LGBM,DMS_plus_sentiment,0.515908,0.528129,30
0,XGB,DMS_only,0.516918,0.528001,25
5,CatBoost+ticker,DMS_plus_fundamentals,0.512606,0.526101,45
12,XGB,DMS_plus_PCA,0.509215,0.524778,89
9,XGB,DMS_plus_sentiment,0.512173,0.520866,30
13,LGBM,DMS_plus_PCA,0.497928,0.520425,89


Play around with sentiment 

In [32]:
from itertools import combinations
import pandas as pd

# Sentiment feature list (should match what you already use)
sent_cols = [
    "mean_sentiment",
    "max_sentiment",
    "min_sentiment",
    "sum_sentiment",
    "news_count",
]

def short_sent_name(col):
    """Optional: shorter labels for the results table."""
    mapping = {
        "mean_sentiment": "mean",
        "max_sentiment": "max",
        "min_sentiment": "min",
        "sum_sentiment": "sum",
        "news_count": "count",
    }
    return mapping.get(col, col)

dms_sent_results = []

# Loop over all non-empty subsets of sentiment features
for k in range(1, len(sent_cols) + 1):
    for subset in combinations(sent_cols, k):
        subset_list = list(subset)
        # Name group like: DMS+mean, DMS+mean+max, etc.
        subset_label = "+".join(short_sent_name(c) for c in subset_list)
        group_name = f"DMS_plus_{subset_label}"

        # Columns used for this experiment
        cols = dms_cols + subset_list

        print(f"\n=== Feature group: {group_name} (k={k}, +{subset_label}) ===")

        # --------- XGBoost (no tic) ---------
        val_xgb, test_xgb = run_xgb_on_cols(cols, group_name)
        dms_sent_results.append({
            "model": "XGB",
            "group": group_name,
            "sent_subset": subset_list,
            "k_sent": k,
            "val_DA": val_xgb,
            "test_DA": test_xgb,
            "n_features": len(cols),
        })

        # --------- LightGBM (no tic; safe names handled inside run_lgb_on_cols) ---------
        val_lgb, test_lgb = run_lgb_on_cols(cols, group_name)
        dms_sent_results.append({
            "model": "LGBM",
            "group": group_name,
            "sent_subset": subset_list,
            "k_sent": k,
            "val_DA": val_lgb,
            "test_DA": test_lgb,
            "n_features": len(cols),
        })

        # --------- CatBoost (with ticker as categorical) ---------
        val_cb, test_cb = run_cat_on_cols(cols, group_name)
        dms_sent_results.append({
            "model": "CatBoost+ticker",
            "group": group_name,
            "sent_subset": subset_list,
            "k_sent": k,
            "val_DA": val_cb,
            "test_DA": test_cb,
            "n_features": len(cols) + 1,  # +1 for tic
        })

# Turn into a DataFrame and sort by val/test DA
dms_sent_results_df = (
    pd.DataFrame(dms_sent_results)
      .sort_values(["val_DA", "test_DA"], ascending=[False, False])
      .reset_index(drop=True)
)

dms_sent_results_df.head(20)


=== Feature group: DMS_plus_mean (k=1, +mean) ===
[XGB] DMS_plus_mean: Val DA=0.5153, Test DA=0.5257
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=

,model,group,sent_subset,k_sent,val_DA,test_DA,n_features
0,LGBM,DMS_plus_min+sum+count,"[min_sentiment, sum_sentiment, news_count]",3,0.518489,0.531753,28
1,LGBM,DMS_plus_mean+max+count,"[mean_sentiment, max_sentiment, news_count]",3,0.517896,0.528202,28
2,LGBM,DMS_plus_mean+sum+count,"[mean_sentiment, sum_sentiment, news_count]",3,0.517744,0.527512,28
3,XGB,DMS_plus_count,[news_count],1,0.517576,0.526726,26
4,LGBM,DMS_plus_max+sum+count,"[max_sentiment, sum_sentiment, news_count]",3,0.517544,0.532307,28
5,LGBM,DMS_plus_mean+max+sum+count,"[mean_sentiment, max_sentiment, sum_sentiment,...",4,0.517295,0.531040,29
6,LGBM,DMS_plus_mean+count,"[mean_sentiment, news_count]",2,0.517199,0.526478,27
7,LGBM,DMS_plus_mean+max,"[mean_sentiment, max_sentiment]",2,0.517111,0.526301,27
8,LGBM,DMS_plus_max+min,"[max_sentiment, min_sentiment]",2,0.517014,0.525628,27
9,XGB,DMS_plus_mean+count,"[mean_sentiment, news_count]",2,0.516966,0.523680,27


In [33]:
dms_sent_results_df.sort_values('test_DA', ascending=False).head(20)

,model,group,sent_subset,k_sent,val_DA,test_DA,n_features
48,LGBM,DMS_plus_mean,[mean_sentiment],1,0.514297,0.539771,26
45,LGBM,DMS_plus_min,[min_sentiment],1,0.514433,0.539338,26
54,LGBM,DMS_plus_sum,[sum_sentiment],1,0.513856,0.538945,26
21,LGBM,DMS_plus_count,[news_count],1,0.515980,0.538929,26
20,LGBM,DMS_plus_max,[max_sentiment],1,0.515988,0.536644,26
36,CatBoost+ticker,DMS_plus_min+count,"[min_sentiment, news_count]",2,0.515235,0.534928,28
65,CatBoost+ticker,DMS_plus_min+sum,"[min_sentiment, sum_sentiment]",2,0.513455,0.534760,28
59,CatBoost+ticker,DMS_plus_mean+max+sum,"[mean_sentiment, max_sentiment, sum_sentiment]",3,0.513672,0.534543,29
68,CatBoost+ticker,DMS_plus_max+min+sum,"[max_sentiment, min_sentiment, sum_sentiment]",3,0.513103,0.534471,29
40,LGBM,DMS_plus_mean+max+min+count,"[mean_sentiment, max_sentiment, min_sentiment,...",4,0.514978,0.534175,29


Play around with fundamentals

In [35]:
from itertools import combinations

best_fund_cols = [
    "log_mktcap",
    "bm",
    "roa_ttm",
    "roe_ttm",
    "cf_yield",
]

fund_combos = []
for r in range(1, len(best_fund_cols)+1):
    for combo in combinations(best_fund_cols, r):
        fund_combos.append(list(combo))

In [37]:
results_fund_phase = []

for combo in fund_combos:
    label = "DMS_plus_" + "_".join(combo)
    cols = dms_cols + combo

    print(f"\n=== Testing {label} ===")

    # XGBoost
    val_xgb, test_xgb = run_xgb_on_cols(cols, label)
    results_fund_phase.append({
        "model": "XGB",
        "group": label,
        "val_DA": val_xgb,
        "test_DA": test_xgb,
        "n_features": len(cols),
    })

    # LightGBM (safe names)
    val_lgb, test_lgb = run_lgb_on_cols(cols, label)
    results_fund_phase.append({
        "model": "LGBM",
        "group": label,
        "val_DA": val_lgb,
        "test_DA": test_lgb,
        "n_features": len(cols),
    })

    # CatBoost (with tic)
    val_cb, test_cb = run_cat_on_cols(cols, label)
    results_fund_phase.append({
        "model": "CatBoost+ticker",
        "group": label,
        "val_DA": val_cb,
        "test_DA": test_cb,
        "n_features": len(cols) + 1,
    })

results_fund_phase_df = pd.DataFrame(results_fund_phase).sort_values(
    ["val_DA", "test_DA"], ascending=[False, False]
)

results_fund_phase_df


=== Testing DMS_plus_log_mktcap ===
[XGB] DMS_plus_log_mktcap: Val DA=0.5165, Test DA=0.5273
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[Lig

,model,group,val_DA,test_DA,n_features
76,LGBM,DMS_plus_log_mktcap_bm_roa_ttm_roe_ttm,0.521792,0.527239,29
64,LGBM,DMS_plus_bm_roa_ttm_roe_ttm,0.521055,0.528290,28
31,LGBM,DMS_plus_bm_roe_ttm,0.520469,0.533341,27
37,LGBM,DMS_plus_roa_ttm_roe_ttm,0.519564,0.533742,27
28,LGBM,DMS_plus_bm_roa_ttm,0.519355,0.533509,27
...,...,...,...,...,...
35,CatBoost+ticker,DMS_plus_bm_cf_yield,0.513672,0.530871,28
23,CatBoost+ticker,DMS_plus_log_mktcap_roe_ttm,0.513648,0.529356,28
68,CatBoost+ticker,DMS_plus_bm_roa_ttm_cf_yield,0.513640,0.530607,29
92,CatBoost+ticker,DMS_plus_log_mktcap_bm_roa_ttm_roe_ttm_cf_yield,0.512998,0.529116,31


In [ ]:
results_fund_phase_df.head(20)

# this uses all the most predictive. so likely some meaning 

,model,group,val_DA,test_DA,n_features
76,LGBM,DMS_plus_log_mktcap_bm_roa_ttm_roe_ttm,0.521792,0.527239,29
64,LGBM,DMS_plus_bm_roa_ttm_roe_ttm,0.521055,0.528290,28
31,LGBM,DMS_plus_bm_roe_ttm,0.520469,0.533341,27
37,LGBM,DMS_plus_roa_ttm_roe_ttm,0.519564,0.533742,27
28,LGBM,DMS_plus_bm_roa_ttm,0.519355,0.533509,27
13,LGBM,DMS_plus_cf_yield,0.519227,0.535233,26
55,LGBM,DMS_plus_log_mktcap_roa_ttm_roe_ttm,0.518986,0.530174,28
10,LGBM,DMS_plus_roe_ttm,0.518858,0.535393,26
52,LGBM,DMS_plus_log_mktcap_bm_cf_yield,0.518826,0.527833,28
18,XGB,DMS_plus_log_mktcap_roa_ttm,0.518618,0.524433,27


In [ ]:
results_fund_phase_df.sort_values('test_DA', ascending=False).head(20)

# this result wouldn't make sense? given that ROA is more predictive. probably more so by chance 

,model,group,val_DA,test_DA,n_features
7,LGBM,DMS_plus_roa_ttm,0.517584,0.536043,26
10,LGBM,DMS_plus_roe_ttm,0.518858,0.535393,26
1,LGBM,DMS_plus_log_mktcap,0.517167,0.535289,26
13,LGBM,DMS_plus_cf_yield,0.519227,0.535233,26
4,LGBM,DMS_plus_bm,0.518073,0.534656,26
88,LGBM,DMS_plus_bm_roa_ttm_roe_ttm_cf_yield,0.517351,0.534239,29
16,LGBM,DMS_plus_log_mktcap_bm,0.517006,0.534094,27
49,LGBM,DMS_plus_log_mktcap_bm_roe_ttm,0.518000,0.533806,28
37,LGBM,DMS_plus_roa_ttm_roe_ttm,0.519564,0.533742,27
11,CatBoost+ticker,DMS_plus_roe_ttm,0.515900,0.533670,27


PCA + sentiment

In [40]:
# Sentiment features
sent_cols = [
    "mean_sentiment", "max_sentiment",
    "min_sentiment", "sum_sentiment", "news_count",
]

# PCA embedding features
pca_cols = [c for c in X_train.columns if c.startswith("pca_emb_")]

# Combine
dms_sent_pca_cols = dms_cols + sent_cols + pca_cols
print("Total features:", len(dms_sent_pca_cols))

Total features: 94


In [44]:
results_pca_sent = []

print("\n=== Feature group: DMS_plus_PCA_plus_sentiment ===")

# ------------------------------------------
# XGBoost (no tic)
# ------------------------------------------
val_xgb, test_xgb = run_xgb_on_cols(
    dms_sent_pca_cols,
    label="DMS + PCA + Sentiment"
)

results_pca_sent.append({
    "model": "XGB",
    "group": "DMS_plus_PCA_plus_sentiment",
    "val_DA": val_xgb,
    "test_DA": test_xgb,
    "n_features": len(dms_sent_pca_cols),
})

# ------------------------------------------
# LightGBM (safe colnames, no tic)
# ------------------------------------------
val_lgb, test_lgb = run_lgb_on_cols(
    dms_sent_pca_cols,
    label="DMS + PCA + Sentiment",
)

results_pca_sent.append({
    "model": "LGBM",
    "group": "DMS_plus_PCA_plus_sentiment",
    "val_DA": val_lgb,
    "test_DA": test_lgb,
    "n_features": len(dms_sent_pca_cols),
})

# ------------------------------------------
# CatBoost (include tic as categorical)
# ------------------------------------------
val_cb, test_cb = run_cat_on_cols(
    dms_sent_pca_cols,
    label="DMS + PCA + Sentiment"
)

results_pca_sent.append({
    "model": "CatBoost+ticker",
    "group": "DMS_plus_PCA_plus_sentiment",
    "val_DA": val_cb,
    "test_DA": test_cb,
    "n_features": len(dms_sent_pca_cols) + 1,  # +1 for tic
})

results_pca_sent_df = pd.DataFrame(results_pca_sent).sort_values(
    ["val_DA", "test_DA"], ascending=[False, False]
)

results_pca_sent_df


=== Feature group: DMS_plus_PCA_plus_sentiment ===
[XGB] DMS + PCA + Sentiment: Val DA=0.5076, Test DA=0.5231
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_

,model,group,val_DA,test_DA,n_features
1,LGBM,DMS_plus_PCA_plus_sentiment,0.513624,0.534319,94
0,XGB,DMS_plus_PCA_plus_sentiment,0.507603,0.523070,94
2,CatBoost+ticker,DMS_plus_PCA_plus_sentiment,0.506994,0.528755,95


PCA + mean sentiment

In [45]:
# ============================
# DMS + PCA + mean_sentiment
# ============================

# 1) Define the column set
dms_pca_mean_cols = dms_cols + pca_cols + ["mean_sentiment"]

print("Num DMS cols:        ", len(dms_cols))
print("Num PCA cols:        ", len(pca_cols))
print("Plus mean_sentiment: ", 1)
print("Total features:      ", len(dms_pca_mean_cols))

results_dms_pca_mean = []

print("\n=== Feature group: DMS + PCA + mean_sentiment ===")

# ---- XGBoost (no tic) ----
val_xgb, test_xgb = run_xgb_on_cols(dms_pca_mean_cols, "DMS+PCA+mean")
results_dms_pca_mean.append({
    "model": "XGB",
    "group": "DMS+PCA+mean",
    "val_DA": val_xgb,
    "test_DA": test_xgb,
    "n_features": len(dms_pca_mean_cols),
})

# ---- LightGBM (no tic; using LGBM-safe column names inside helper) ----
val_lgb, test_lgb = run_lgb_on_cols(dms_pca_mean_cols, "DMS+PCA+mean")
results_dms_pca_mean.append({
    "model": "LGBM",
    "group": "DMS+PCA+mean",
    "val_DA": val_lgb,
    "test_DA": test_lgb,
    "n_features": len(dms_pca_mean_cols),
})

# ---- CatBoost (with ticker as categorical, handled inside helper) ----
val_cb, test_cb = run_cat_on_cols(dms_pca_mean_cols, "DMS+PCA+mean")
results_dms_pca_mean.append({
    "model": "CatBoost+ticker",
    "group": "DMS+PCA+mean",
    "val_DA": val_cb,
    "test_DA": test_cb,
    "n_features": len(dms_pca_mean_cols) + 1,  # +1 for tic
})

results_dms_pca_mean_df = pd.DataFrame(results_dms_pca_mean)
results_dms_pca_mean_df

Num DMS cols:         25
Num PCA cols:         64
Plus mean_sentiment:  1
Total features:       90

=== Feature group: DMS + PCA + mean_sentiment ===
[XGB] DMS+PCA+mean: Val DA=0.5060, Test DA=0.5268
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [War

,model,group,val_DA,test_DA,n_features
0,XGB,DMS+PCA+mean,0.506032,0.526758,90
1,LGBM,DMS+PCA+mean,0.490898,0.518003,90
2,CatBoost+ticker,DMS+PCA+mean,0.502417,0.525917,91


In [ ]:
results_dms_pca_mean_df.sort_values('val_DA', ascending=False)

# just using mean is worse than using all of sentiment 

,model,group,val_DA,test_DA,n_features
0,XGB,DMS+PCA+mean,0.506032,0.526758,90
2,CatBoost+ticker,DMS+PCA+mean,0.502417,0.525917,91
1,LGBM,DMS+PCA+mean,0.490898,0.518003,90


PCA + each sentiment

In [53]:
# =====================================================
# 4. Loop through EACH sentiment feature
# =====================================================

for s in sent_cols:
    print(f"\n==============================")
    print(f"=== DMS + PCA + {s} ===")
    print("==============================")

    # The combined feature list
    combined_cols = dms_cols + pca_cols + [s]

    # -------------------------------------------------
    #  XGBoost (no tic)
    # -------------------------------------------------
    val_xgb, test_xgb = run_xgb_on_cols(
        col_list=combined_cols,
        label=f"DMS+PCA+{s}"
    )
    results_pca_sent.append({
        "model": "XGB",
        "sent_feature": s,
        "val_DA": val_xgb,
        "test_DA": test_xgb,
        "n_features": len(combined_cols),
    })

    # -------------------------------------------------
    # LightGBM (needs safe names)
    # -------------------------------------------------
    val_lgb, test_lgb = run_lgb_on_cols(
        col_list=combined_cols,
        label=f"DMS+PCA+{s}"
    )
    results_pca_sent.append({
        "model": "LGBM",
        "sent_feature": s,
        "val_DA": val_lgb,
        "test_DA": test_lgb,
        "n_features": len(combined_cols),
    })

    # -------------------------------------------------
    # CatBoost (with tic)
    # -------------------------------------------------
    val_cb, test_cb = run_cat_on_cols(
        col_list=combined_cols,
        label=f"DMS+PCA+{s}"
    )
    results_pca_sent.append({
        "model": "CatBoost+ticker",
        "sent_feature": s,
        "val_DA": val_cb,
        "test_DA": test_cb,
        "n_features": len(combined_cols) + 1,
    })

# =====================================================
# 5. Convert results into a table
# =====================================================

results_pca_sent_df = (
    pd.DataFrame(results_pca_sent)
      .sort_values(["val_DA", "test_DA"], ascending=[False, False])
      .reset_index(drop=True)
)

results_pca_sent_df


=== DMS + PCA + mean_sentiment ===
[XGB] DMS+PCA+mean_sentiment: Val DA=0.5060, Test DA=0.5268
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[L

,model,group,val_DA,test_DA,n_features,sent_feature
0,LGBM,DMS_plus_PCA_plus_sentiment,0.513624,0.534319,94,NaN
1,XGB,NaN,0.508854,0.527993,90,news_count
2,XGB,NaN,0.508814,0.528202,90,sum_sentiment
3,XGB,NaN,0.507780,0.528546,90,max_sentiment
4,XGB,DMS_plus_PCA_plus_sentiment,0.507603,0.523070,94,NaN
5,CatBoost+ticker,NaN,0.507355,0.528506,91,max_sentiment
6,XGB,NaN,0.507355,0.528346,90,min_sentiment
7,CatBoost+ticker,NaN,0.507026,0.529156,91,news_count
8,CatBoost+ticker,NaN,0.507002,0.527897,91,sum_sentiment
9,CatBoost+ticker,DMS_plus_PCA_plus_sentiment,0.506994,0.528755,95,NaN


Using sentiment and predictive fundamentals

In [47]:
best_fund_cols = [
    "log_mktcap",
    "bm",
    "roa_ttm",
    "roe_ttm",
    "cf_yield",
]

sent_cols = [
    "mean_sentiment",
    "max_sentiment",
    "min_sentiment",
    "sum_sentiment",
    "news_count",
]

In [50]:
# ======================================================
# 1. Define combined feature set: DMS + top fundamentals + sentiment
# ======================================================

combined_cols = dms_cols + best_fund_cols + sent_cols
print("Total combined features:", len(combined_cols))

# Safety check: ensure all columns exist in X_train
missing_cols = [c for c in combined_cols if c not in X_train.columns]
if missing_cols:
    print("WARNING: Missing columns:", missing_cols)
else:
    print("All columns found.")


# ======================================================
# 2. Run XGBoost (no tic)
# ======================================================
print("\n=== XGBoost: DMS + top fundamentals + sentiment ===")
val_xgb, test_xgb = run_xgb_on_cols(
    combined_cols,
    label="DMS+funds+sent"
)
print("XGB → Val:", val_xgb, " Test:", test_xgb)


# ======================================================
# 3. Run LightGBM (no tic)
#    Must map to safe names first
# ======================================================

print("\n=== LightGBM: DMS + top fundamentals + sentiment ===")
val_lgb, test_lgb = run_lgb_on_cols(
    combined_cols,
    label="DMS+funds+sent",
)
print("LGBM → Val:", val_lgb, " Test:", test_lgb)


# ======================================================
# 4. Run CatBoost (with ticker categorical)
#    Cat model auto-handles tic as category
# ======================================================
print("\n=== CatBoost+ticker: DMS + top fundamentals + sentiment ===")
val_cb, test_cb = run_cat_on_cols(
    combined_cols,
    label="DMS+funds+sent"
)
print("CatBoost → Val:", val_cb, " Test:", test_cb)


# ======================================================
# 5. Combine results in a table
# ======================================================
final_results = pd.DataFrame([
    {"Model": "XGB", "Val_DA": val_xgb, "Test_DA": test_xgb},
    {"Model": "LGBM", "Val_DA": val_lgb, "Test_DA": test_lgb},
    {"Model": "CatBoost+ticker", "Val_DA": val_cb, "Test_DA": test_cb},
])

final_results

Total combined features: 35
All columns found.

=== XGBoost: DMS + top fundamentals + sentiment ===
[XGB] DMS+funds+sent: Val DA=0.5142, Test DA=0.5223
XGB → Val: 0.5141606611782247  Test: 0.5223087222493926

=== LightGBM: DMS + top fundamentals + sentiment ===
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will b

,Model,Val_DA,Test_DA
0,XGB,0.514161,0.522309
1,LGBM,0.510818,0.529084
2,CatBoost+ticker,0.508838,0.528931


In [51]:
# ============================================
# DMS + best fundamentals + each sentiment feature
# ============================================

best_fund_cols = [
    "log_mktcap",
    "bm",
    "roa_ttm",
    "roe_ttm",
    "cf_yield",
]

sent_cols = [
    "mean_sentiment",
    "max_sentiment",
    "min_sentiment",
    "sum_sentiment",
    "news_count",
]

fund_sent_results = []

for s in sent_cols:
    combo_name   = f"DMS+bestFund+{s}"
    combined_cols = dms_cols + best_fund_cols + [s]

    print(f"\n=== Feature group: {combo_name} ===")
    print(f"#features: {len(combined_cols)}")

    # 1) XGBoost (no tic)
    val_xgb, test_xgb = run_xgb_on_cols(combined_cols, combo_name)
    fund_sent_results.append({
        "model": "XGB",
        "group": combo_name,
        "val_DA": val_xgb,
        "test_DA": test_xgb,
        "n_features": len(combined_cols),
    })

    # 2) LightGBM (no tic; run_lgb_on_cols should handle safe colnames internally)
    val_lgb, test_lgb = run_lgb_on_cols(combined_cols, combo_name)
    fund_sent_results.append({
        "model": "LGBM",
        "group": combo_name,
        "val_DA": val_lgb,
        "test_DA": test_lgb,
        "n_features": len(combined_cols),
    })

    # 3) CatBoost (with tic as categorical, handled inside run_cat_on_cols)
    val_cb, test_cb = run_cat_on_cols(combined_cols, combo_name)
    fund_sent_results.append({
        "model": "CatBoost+ticker",
        "group": combo_name,
        "val_DA": val_cb,
        "test_DA": test_cb,
        "n_features": len(combined_cols) + 1,  # +1 for tic
    })

fund_sent_results_df = (
    pd.DataFrame(fund_sent_results)
      .sort_values(["val_DA", "test_DA"], ascending=[False, False])
)

fund_sent_results_df


=== Feature group: DMS+bestFund+mean_sentiment ===
#features: 31
[XGB] DMS+bestFund+mean_sentiment: Val DA=0.5157, Test DA=0.5227
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Cur

,model,group,val_DA,test_DA,n_features
4,LGBM,DMS+bestFund+max_sentiment,0.519652,0.526983,31
13,LGBM,DMS+bestFund+news_count,0.518866,0.525917,31
1,LGBM,DMS+bestFund+mean_sentiment,0.517303,0.524457,31
10,LGBM,DMS+bestFund+sum_sentiment,0.516998,0.524602,31
3,XGB,DMS+bestFund+max_sentiment,0.516133,0.522493,31
9,XGB,DMS+bestFund+sum_sentiment,0.516044,0.521531,31
12,XGB,DMS+bestFund+news_count,0.515876,0.521884,31
0,XGB,DMS+bestFund+mean_sentiment,0.515692,0.522661,31
6,XGB,DMS+bestFund+min_sentiment,0.515283,0.521523,31
7,LGBM,DMS+bestFund+min_sentiment,0.515002,0.530214,31


In [52]:
fund_sent_results_df.sort_values('test_DA', ascending=False)

,model,group,val_DA,test_DA,n_features
5,CatBoost+ticker,DMS+bestFund+max_sentiment,0.513752,0.530446,32
7,LGBM,DMS+bestFund+min_sentiment,0.515002,0.530214,31
2,CatBoost+ticker,DMS+bestFund+mean_sentiment,0.512381,0.529981,32
11,CatBoost+ticker,DMS+bestFund+sum_sentiment,0.512349,0.528979,32
14,CatBoost+ticker,DMS+bestFund+news_count,0.511387,0.528434,32
8,CatBoost+ticker,DMS+bestFund+min_sentiment,0.512710,0.527528,32
4,LGBM,DMS+bestFund+max_sentiment,0.519652,0.526983,31
13,LGBM,DMS+bestFund+news_count,0.518866,0.525917,31
10,LGBM,DMS+bestFund+sum_sentiment,0.516998,0.524602,31
1,LGBM,DMS+bestFund+mean_sentiment,0.517303,0.524457,31
